# 04 — Land Surface Preprocessing

Inventories and validates DEM, Slope, Aspect, NDVI, LST Day and
Distance-to-Sea predictor files.

In [2]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [3]:
import pandas as pd
import rasterio

predictor_root = RAW_DIR / "predictors"
raster_files = sorted([
    *predictor_root.rglob("*.tif"),
    *predictor_root.rglob("*.tiff"),
])

if not raster_files:
    raise FileNotFoundError("No predictor GeoTIFF files were found.")

records = []
for path in raster_files:
    with rasterio.open(path) as src:
        records.append({
            "predictor": path.parent.name,
            "file": path.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution_x": src.res[0],
            "resolution_y": src.res[1],
            "nodata": src.nodata,
            "dtype": src.dtypes[0],
        })

predictor_inventory = pd.DataFrame(records)
display(predictor_inventory)

,predictor,file,crs,width,height,resolution_x,resolution_y,nodata,dtype
0,Aspect,Aspect_Degree_30m.tif,"PROJCS[""WGS 84 / UTM zone 45N"",GEOGCS[""WGS 84""...",1755,4974,30.000000,30.000000,NaN,float32
1,DEM,Khulna_SRTM_DEM.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",1929,5004,0.000269,0.000269,NaN,int16
2,Distance_Sea,Distance_Sea.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,31,0.050000,0.050000,-3.402823e+38,float32
3,Distance_Sea,Distance_Sea_clip.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,30,5394.264043,5394.264043,-3.402823e+38,float32
4,Distance_Sea,Distance_Sea_clip2.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,29,5394.264043,5394.264043,-3.402823e+38,float32
...,...,...,...,...,...,...,...,...,...
147,NDVI,NDVI_2022_6.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,float64
148,NDVI,NDVI_2022_7.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,float64
149,NDVI,NDVI_2022_8.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,float64
150,NDVI,NDVI_2022_9.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,float64


In [4]:
output_dir = INTERIM_DIR / "inventories"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "predictor_raster_inventory.csv"
predictor_inventory.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\interim\inventories\predictor_raster_inventory.csv
